# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [3]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [4]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [5]:
# openai = OpenAI()

ADESSO_BASE_URL = os.getenv('ADESSO_BASE_URL')
vultr_base_url = os.getenv('VULTR_BASE_URL')
adesso_sovereign_ai_hub_key = os.getenv('ADESSO_SOVEREIGN_AI_HUB_KEY')
adesso_api_key = os.getenv('ADESSO_API_KEY')
vultr_api_key = os.getenv('VULTR_API_KEY')

adesso = OpenAI(
    base_url=ADESSO_BASE_URL,
    api_key=adesso_sovereign_ai_hub_key
)

vultr = OpenAI(
    api_key=vultr_api_key, 
    base_url=vultr_base_url
)

adesso_premium = OpenAI(
    base_url=ADESSO_BASE_URL,
    api_key=adesso_api_key
)

model_name = os.getenv('FREE_DEFAULT_MODEL')

In [6]:
# Some lists!

todos = []
completed = []

In [7]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [8]:
get_todo_report()

''

In [9]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [10]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [11]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [12]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    done = False
    while not done:
        response = vultr.chat.completions.create(model="nvidia/DeepSeek-V3.2-NVFP4", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [39]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [40]:
todos, completed = [], []
loop(messages)

Todo #1: Establish the distance between Boston and New York (by rail)
Todo #2: Calculate how far the Boston train travels before the NY train departs
Todo #3: Calculate remaining distance and combined closing speed
Todo #4: Determine the meeting time

The standard rail distance between Boston and New York is approximately 215 miles. This is the commonly used 
distance for train problems involving these cities.

Todo #1: Establish the distance between Boston and New York (by rail)
Todo #2: Calculate how far the Boston train travels before the NY train departs
Todo #3: Calculate remaining distance and combined closing speed
Todo #4: Determine the meeting time

The Boston train leaves at 2:00 PM and the NY train leaves at 3:00 PM — a 1-hour head start. In that 1 hour, the 
Boston train covers: 60 mph × 1 hr = 60 miles.

Todo #1: Establish the distance between Boston and New York (by rail)
Todo #2: Calculate how far the Boston train travels before the NY train departs
Todo #3: Calculate remaining distance and combined closing speed
Todo #4: Determine the meeting time

Remaining distance at 3:00 PM: 215 - 60 = 155 miles. Both trains now move toward each other, so their combined 
closing speed is: 60 + 80 = 140 mph.

Todo #1: Establish the distance between Boston and New York (by rail)
Todo #2: Calculate how far the Boston train travels before the NY train departs
Todo #3: Calculate remaining distance and combined closing speed
Todo #4: Determine the meeting time

Time after 3:00 PM until they meet: 155 ÷ 140 = 31/28 hours ≈ 1 hour, 6 minutes, and 26 seconds. So they meet at 
approximately 4:06 PM (and ~26 seconds).

Todo #1: Establish the distance between Boston and New York (by rail)
Todo #2: Calculate how far the Boston train travels before the NY train departs
Todo #3: Calculate remaining distance and combined closing speed
Todo #4: Determine the meeting time

## 🚂 Solution: When Do the Trains Meet?

### Given Information
- **Boston train**: Departs 2:00 PM, travels at **60 mph** toward New York
- **New York train**: Departs 3:00 PM, travels at **80 mph** toward Boston
- **Distance (by rail)**: ~**215 miles** (standard Boston–NYC rail distance)

---

### Step 1 — Boston Train's Head Start
From 2:00 PM to 3:00 PM, the Boston train travels alone:

> 60 mph × 1 hr = **60 miles**

### Step 2 — Remaining Distance at 3:00 PM
> 215 miles − 60 miles = **155 miles** still separate the two trains

### Step 3 — Closing Speed
Both trains now head toward each other:

> 60 mph + 80 mph = **140 mph** combined closing speed

### Step 4 — Time to Meet (after 3:00 PM)
> 155 miles ÷ 140 mph = **31/28 hours ≈ 1 hour, 6 min, 26 sec**

---

### ✅ Answer

The two trains meet at approximately **4:06 PM** (roughly 4:06 and 26 seconds).

At that moment, the Boston train will have traveled about **126 miles** from Boston, and the New York train will 
have traveled about **89 miles** from New York (126 + 89 = 215 ✓).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>